# V14b — logit-margin occlusion, logit predictions, logit region occlusion (patch for V14)
**GPU (T4 or P100), about 4–5 h, inference only.** Inputs: exactly the same 5 inputs as V14
(V12 run-2 output + the four V13 datasets with `ckpt/`). The log must say `36 checkpoints found`.

**Why:** V14 measured occlusion as a drop in softmax *probability*. For very confident models the probability is exactly 1.0
in floating point, so no patch changes it and the map is empty (lfr = NaN). This removed up to 29 % of images, non-randomly
(the most confident true positives), and differently per architecture and arm. V13 predictions and V14 part C also stored
probabilities, which creates tied scores at 0/1. V14b uses the **logit margin** (logit_pneumonia − logit_normal), which does not
saturate. Grad-CAM, rollout and part B of V14 are not affected and are reused as they are.

| Part | Models | Images | Output |
|---|---|---|---|
| 0 | all 36 | val, internal test, RSNA, NIH, VinDr | `preds_logit/{arch}__{arm}__s{seed}.npz` (logit margins) |
| A | 12 seed-42 | the same grounding sample as V14 | occlusion maps on the fp32 logit margin → `grounding/A14b__*.csv` (method `occlusion_logit`) |
| C | 12 seed-42 | full cohorts, same random regions as V14 | `occlusion/occ14b__*.npz` with `{cohort}__logit` |

Resume-safe: each model writes its own file; finished files are skipped.

In [ ]:
import os, json, glob, time, math, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
import timm
from sklearn.metrics import roc_auc_score
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
INPUT = os.environ.get("V14_INPUT", "/kaggle/input")
OUT = os.environ.get("V14_OUT", "/kaggle/working")
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
ARCHS = ["vit_base_patch16_224", "densenet121", "convnext_tiny", "swin_tiny_patch4_window7_224"]
ARMS, SEEDS = ["raw", "roi", "lung"], [42, 1337, 2024]
N_POS, N_NEG, N_NOBOX, N_SEEDVAR = (3, 2, 4, 3) if QUICK_TEST else (200, 100, 300, 100)
GRID = 4 if QUICK_TEST else 14                              # occlusion-sensitivity grid (16-px patches at 14)
TAUS, POINT_TOL = (0.0, 0.1, 0.2, 0.4), 8
OCC_BATCH = 16 if QUICK_TEST else 98
S, MEAN, STD, FILL = 224, 0.4769, 0.2414, 122
for d in ("grounding", "occlusion", "maps", "preds_logit"): os.makedirs(os.path.join(OUT, d), exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
T0 = time.time()

def find_file(name):
    for r, _, fs in os.walk(INPUT):
        if name in fs: return os.path.join(r, name)
    return None

## 1. Cache, samples and checkpoints

In [ ]:
MANIFEST = find_file("manifest_v2.csv"); assert MANIFEST, "add the V12 output"
CDIR = os.path.dirname(MANIFEST)
meta = pd.read_csv(MANIFEST)
cache = {}
for c in meta.cohort.unique():
    z = np.load(os.path.join(CDIR, f"cache_{c}.npz"))
    cache[c] = {k: z[k] for k in z.files}
chance = pd.read_csv(os.path.join(CDIR, "chance_levels_v2.csv"))
EVAL = {"internal": meta[(meta.cohort == "internal") & (meta.split == "test")]}
for c in ("rsna", "nih", "vindr"):
    if c in cache: EVAL[c] = meta[meta.cohort == c]
if QUICK_TEST:
    EVAL = {c: d.groupby("label", group_keys=False).head(5) for c, d in EVAL.items()}

def grounding_sample(d):
    r = np.random.default_rng(SEED)
    pos_box = d[(d.label == 1) & (d.n_boxes > 0)]
    if len(pos_box):
        p = pos_box.iloc[r.permutation(len(pos_box))[:N_POS]]
        neg = d[d.label == 0]; n = neg.iloc[r.permutation(len(neg))[:N_NEG]]
        return pd.concat([p, n])
    half = N_NOBOX // 2
    return pd.concat([d[d.label == k].iloc[r.permutation(int((d.label == k).sum()))[:half]] for k in (1, 0)])
GSAMPLE = {c: grounding_sample(d) for c, d in EVAL.items()}
SEEDVAR = {c: g.iloc[np.random.default_rng(SEED + 1).permutation(len(g))[:N_SEEDVAR]] for c, g in GSAMPLE.items()}
print({c: (len(g), int(g.label.sum())) for c, g in GSAMPLE.items()})

CKPT = {os.path.basename(p)[:-4]: p for p in glob.glob(os.path.join(INPUT, "**", "ckpt", "*.pth"), recursive=True)}
print(len(CKPT), "checkpoints found")

def load_model(arch, arm, seed):
    key = f"{arch}__{arm}__s{seed}"
    if key not in CKPT: return None
    m = timm.create_model(arch, pretrained=False, num_classes=2)
    m.load_state_dict(torch.load(CKPT[key], map_location="cpu")["state_dict"])
    return m.to(device).eval()

def lung_def(arm, m):
    """One lung definition in every frame: the dilated lung mask (the region arm C can see). From the 2026-10-17 V12 onwards the
    stored masks are already dilated in all three arms, so this is the identity; kept as the single place to change it."""
    return m.astype(np.uint8)

def to_tensor(u8):                                            # (N,224,224) uint8 -> (N,3,224,224) normalised
    x = torch.from_numpy(np.ascontiguousarray(u8)).float().div(255.0).sub(MEAN).div(STD)
    return x.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()

@torch.no_grad()
def probs(model, u8, bs=128):
    out = []
    for i in range(0, len(u8), bs):
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            out.append(torch.softmax(model(to_tensor(u8[i:i + bs]).to(device)).float(), 1).cpu())
    return torch.cat(out).numpy()

## 2. Logit margin and logit-based occlusion map

In [ ]:
@torch.no_grad()
def margins(model, u8, bs=128, fp32=False):
    """Logit margin (pneumonia − normal) = log-odds; no saturation. fp32=True: whole network in fp32 (used for the maps,
    where patch effects are small). Otherwise the backbone runs in fp16 and the head in fp32 (used for AUC ranking)."""
    out = []
    for i in range(0, len(u8), bs):
        x = to_tensor(u8[i:i + bs]).to(device)
        if fp32 or device.type != "cuda":
            l = model(x).float()
        else:
            with torch.autocast("cuda", dtype=torch.float16):
                f = model.forward_features(x)
            l = model.forward_head(f.float()).float()
        out.append((l[:, 1] - l[:, 0]).double().cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def occlusion_map_logit(model, img_u8):
    """Evidence for the predicted class: decrease of its log-odds when each 16-px patch is set to mean grey."""
    m0 = float(margins(model, img_u8[None], fp32=True)[0]); c = int(m0 > 0)
    ps = S // GRID; variants = []
    for gy in range(GRID):
        for gx in range(GRID):
            v = img_u8.copy(); v[gy * ps:(gy + 1) * ps, gx * ps:(gx + 1) * ps] = FILL; variants.append(v)
    mv = margins(model, np.stack(variants), bs=OCC_BATCH, fp32=True)
    ev = (m0 - mv) if c == 1 else (mv - m0)
    m = np.clip(ev, 0, None).reshape(GRID, GRID).astype(np.float32)
    return cv2.resize(m, (S, S), interpolation=cv2.INTER_LINEAR), c, m0, int((m > 0).sum())

## Grounding metrics (identical to V14 §3)

In [ ]:
def norm01(s):
    s = s - s.min(); return s / s.max() if s.max() > 1e-12 else s

def metrics(sal, lung, box, tau):
    s = norm01(sal); L = lung.astype(bool); B = box.astype(bool) if box is not None else None
    w = s * (s > tau)
    tot = w.sum(); out = dict(lfr=float((w * L).sum() / tot) if tot > 0 else np.nan)
    if B is not None and B.any():
        wl = (w * L).sum()
        out["lefr"] = float((w * B).sum() / tot) if tot > 0 else np.nan
        out["lefr_in_lung"] = float((w * B * L).sum() / wl) if wl > 0 else np.nan
        if tau == 0.2:
            iy, ix = np.unravel_index(np.argmax(s), s.shape)
            out["point_hit"] = float(B[max(iy - POINT_TOL, 0):iy + POINT_TOL + 1, max(ix - POINT_TOL, 0):ix + POINT_TOL + 1].any())
            ds = cv2.resize(s, (112, 112), interpolation=cv2.INTER_AREA)
            db = cv2.resize(B.astype(np.uint8), (112, 112), interpolation=cv2.INTER_NEAREST)
            out["box_auroc"] = float(roc_auc_score(db.ravel(), ds.ravel())) if 0 < db.sum() < db.size else np.nan
            Bd = cv2.dilate(B.astype(np.uint8), np.ones((2 * POINT_TOL + 1,) * 2, np.uint8)).astype(bool)
            out["point_chance"] = float(Bd.mean())
    return out

## 3. Part 0 — logit predictions for all 36 models (replaces V13 probabilities in the statistics)

In [ ]:
PRED_SETS = {"val": meta[(meta.cohort == "internal") & (meta.split == "val")], "test": EVAL["internal"]}
for c in ("rsna", "nih", "vindr"):
    if c in EVAL: PRED_SETS[c] = EVAL[c]
if QUICK_TEST: PRED_SETS = {c: d.groupby("label", group_keys=False).head(5) for c, d in PRED_SETS.items()}
CACHE_OF = {"val": "internal", "test": "internal"}
for arch in ARCHS:
    for arm in ARMS:
        for seed in SEEDS:
            fn = os.path.join(OUT, "preds_logit", f"{arch}__{arm}__s{seed}.npz")
            if os.path.exists(fn): continue
            model = load_model(arch, arm, seed)
            if model is None: print("  missing checkpoint", arch, arm, seed); continue
            blob, msg = {}, []
            for c, d in PRED_SETS.items():
                imgs = cache[CACHE_OF.get(c, c)][arm]
                mg = margins(model, imgs[d.row.astype(int).values])
                blob[f"{c}__logit"] = mg; blob[f"{c}__y"] = d.label.values.astype(np.int8); blob[f"{c}__id"] = d.image_id.astype(str).values
                if d.label.nunique() == 2: msg.append(f"{c}={roc_auc_score(d.label.values, mg):.4f}")
            np.savez_compressed(fn, **blob)
            del model
            if device.type == "cuda": torch.cuda.empty_cache()
            print(f"  P0 {arch} {arm} s{seed}: AUC(logit) " + " ".join(msg) + f"  ({(time.time() - T0) / 60:.1f} min)")

## 4. Part A — logit-margin occlusion maps on the seed-42 models (same grounding sample as V14)

In [ ]:
AGG = {}
for arch in ARCHS:
    for arm in ARMS:
        fn = os.path.join(OUT, "grounding", f"A14b__{arch}__{arm}__s42.csv")
        if os.path.exists(fn): continue
        model = load_model(arch, arm, 42)
        if model is None: print("  missing checkpoint", arch, arm, 42); continue
        rows, n_empty = [], 0
        for c, g in GSAMPLE.items():
            imgs, lungs, boxes = cache[c][arm], cache[c]["m_" + arm], cache[c]["b_" + arm]
            for _, r in g.iterrows():
                k = int(r.row); img = imgs[k]; box = boxes[k] if r.n_boxes > 0 else None
                sal, pred, m0, npos = occlusion_map_logit(model, img); n_empty += npos == 0
                L_k = lung_def(arm, lungs[k]).astype(bool)
                ch_lung = float(L_k.mean())
                ch_les = float((box.astype(bool) & L_k).sum() / max(L_k.sum(), 1)) if box is not None else np.nan
                a = AGG.setdefault((arch, arm, c, "occlusion_logit", int(r.label)), [np.zeros((S, S)), 0]); a[0] += norm01(sal); a[1] += 1
                for tau in TAUS:
                    rows.append(dict(arch=arch, arm=arm, seed=42, cohort=c, image_id=r.image_id, label=int(r.label), pred=pred,
                                     logit_margin=m0, p_pneu=float(1 / (1 + np.exp(-m0))), n_pos_patches=npos,
                                     method="occlusion_logit", tau=tau, lung_chance=ch_lung, lesion_in_lung_chance=ch_les,
                                     **metrics(sal, L_k, box, tau)))
        pd.DataFrame(rows).to_csv(fn, index=False)
        del model
        if device.type == "cuda": torch.cuda.empty_cache()
        print(f"  A14b {arch} {arm}: {len(rows)} rows, empty maps {n_empty}/{len(rows) // len(TAUS)}  ({(time.time() - T0) / 60:.1f} min)")
if AGG:
    np.savez_compressed(os.path.join(OUT, "maps", "aggregate_maps_occlusion_logit.npz"),
                        **{"|".join(map(str, k)): v[0] / max(v[1], 1) for k, v in AGG.items()})

## 5. Part C — causal region occlusion on the logit margin (same regions and random draws as V14), seed-42 models
Lesion box: positives occlude their radiologist box; negatives occlude a box whose size is drawn from the positive
boxes of the same cohort and arm, placed at random inside the lung (symmetric perturbation, design §6).
Lesion control: a same-size box at a random in-lung position not overlapping the lesion box.

In [ ]:
CORNER_F, BORDER_F, SUBDIA_F, UPPER_F, CENTER_F = 0.22, 0.08, 0.28, 0.20, 0.20     # v1 values (V9)
def geom_masks(Sz):
    m = {}; c = int(round(Sz * CORNER_F)); a = np.zeros((Sz, Sz), bool)
    a[:c, :c] = a[:c, -c:] = a[-c:, :c] = a[-c:, -c:] = True; m["corners"] = a
    b = int(round(Sz * BORDER_F)); a = np.zeros((Sz, Sz), bool); a[:b, :] = a[-b:, :] = a[:, :b] = a[:, -b:] = True; m["border"] = a
    a = np.zeros((Sz, Sz), bool); a[int(round(Sz * (1 - SUBDIA_F))):, :] = True; m["subdiaphragm"] = a
    a = np.zeros((Sz, Sz), bool); a[:int(round(Sz * UPPER_F)), :] = True; m["upper"] = a
    a = np.zeros((Sz, Sz), bool); lo, hi = int(round(Sz * (0.5 - CENTER_F / 2))), int(round(Sz * (0.5 + CENTER_F / 2))); a[:, lo:hi] = True
    m["center"] = a
    return m
GEOM = geom_masks(S)

def random_rect_mask(Sz, area_frac, rng):                      # v1 function (V9)
    area_frac = float(np.clip(area_frac, 0.01, 0.95)); target = area_frac * Sz * Sz
    ar = np.exp(rng.uniform(np.log(0.5), np.log(2.0)))
    h = int(np.clip(round(np.sqrt(target / ar)), 1, Sz)); w = int(np.clip(round(target / max(h, 1)), 1, Sz))
    y = rng.integers(0, Sz - h + 1); x = rng.integers(0, Sz - w + 1)
    a = np.zeros((Sz, Sz), bool); a[y:y + h, x:x + w] = True; return a

def bbox_of(m):
    ys, xs = np.where(m); return (ys.min(), ys.max() + 1, xs.min(), xs.max() + 1) if len(ys) else None

def place_in_lung(h, w, lung, avoid, rng, tries=200):
    """Random h×w rectangle whose centre lies in the lung and which does not overlap `avoid`."""
    ys, xs = np.where(lung)
    for _ in range(tries):
        if not len(ys): break
        j = rng.integers(len(ys)); y0 = int(np.clip(ys[j] - h // 2, 0, S - h)); x0 = int(np.clip(xs[j] - w // 2, 0, S - w))
        a = np.zeros((S, S), bool); a[y0:y0 + h, x0:x0 + w] = True
        if avoid is None or not (a & avoid).any(): return a
    return None

COND = ["baseline", "lungs", "non_lungs", "corners", "border", "subdiaphragm", "upper", "center",
        "rand_lungs", "rand_20", "rand_30", "lesion", "lesion_ctrl"]

def occlusion_variants(img, lung, box, size_pool, rng):
    L = lung.astype(bool)
    masks = {"baseline": np.zeros((S, S), bool), "lungs": L, "non_lungs": ~L, **GEOM,
             "rand_lungs": random_rect_mask(S, max(L.mean(), 0.02), rng),
             "rand_20": random_rect_mask(S, 0.20, rng), "rand_30": random_rect_mask(S, 0.30, rng)}
    les = box.astype(bool) if box is not None and box.any() else None
    if les is None and size_pool:                               # negative (or box-less positive): sampled size in lung
        h, w = size_pool[rng.integers(len(size_pool))]; les = place_in_lung(h, w, L, None, rng)
    ctrl = None
    if les is not None:
        bb = bbox_of(les); ctrl = place_in_lung(bb[1] - bb[0], bb[3] - bb[2], L, les, rng)
    masks["lesion"] = les if les is not None else np.zeros((S, S), bool)
    masks["lesion_ctrl"] = ctrl if ctrl is not None else np.zeros((S, S), bool)
    out = []
    for c in COND:
        v = img.copy(); v[masks[c]] = FILL; out.append(v)
    return out, {c: float(masks[c].mean()) for c in COND}, les is not None and ctrl is not None

for arch in ARCHS:
    for arm in ARMS:
        fn = os.path.join(OUT, "occlusion", f"occ14b__{arch}__{arm}__s42.npz")
        if os.path.exists(fn): continue
        model = load_model(arch, arm, 42)
        if model is None: continue
        blob = {}
        for c, d in EVAL.items():
            rng = np.random.default_rng(SEED)                   # same random regions for every model
            imgs, lungs, boxes = cache[c][arm], cache[c]["m_" + arm], cache[c]["b_" + arm]
            pool = []
            for k in d[(d.label == 1) & (d.n_boxes > 0)].row:
                bb = bbox_of(boxes[int(k)].astype(bool))
                if bb: pool.append((bb[1] - bb[0], bb[3] - bb[2]))
            P = np.zeros((len(d), len(COND)), np.float64); A = np.zeros((len(d), len(COND)), np.float32); ok = np.zeros(len(d), bool)
            for i, k in enumerate(d.row.astype(int)):
                v, area, ok[i] = occlusion_variants(imgs[k], lung_def(arm, lungs[k]), boxes[k] if d.iloc[i].n_boxes > 0 else None, pool, rng)
                P[i] = margins(model, np.stack(v)); A[i] = [area[cc] for cc in COND]
            blob[f"{c}__logit"] = P; blob[f"{c}__area"] = A; blob[f"{c}__lesion_ok"] = ok
            blob[f"{c}__y"] = d.label.values.astype(np.int8); blob[f"{c}__id"] = d.image_id.astype(str).values
            y = d.label.values
            if len(np.unique(y)) == 2:
                msg = " ".join(f"{cc}={roc_auc_score(y, P[:, j]):.3f}" for j, cc in enumerate(COND)
                               if cc in ("baseline", "lungs", "non_lungs", "lesion", "lesion_ctrl"))
                print(f"  occ {arch} {arm} {c}: {msg}")
        blob["conditions"] = np.array(COND)
        np.savez_compressed(fn, **blob)
        del model
        if device.type == "cuda": torch.cuda.empty_cache()

## 6. Consolidate and checks

In [ ]:
G = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(OUT, "grounding", "A14b__*.csv"))], ignore_index=True)
G.to_csv(os.path.join(OUT, "grounding_per_image_v14b.csv"), index=False)
g2 = G[G.tau == 0.2]
print("CHECK empty occlusion maps (lfr NaN) per model — expect ~0:")
print(g2.assign(nan=g2.lfr.isna()).groupby(["arch", "arm"]).nan.mean().unstack().round(3).to_string())
print(g2.groupby(["arch", "arm", "cohort"]).agg(n=("lfr", "size"), lfr=("lfr", "median"), lefr_in_lung=("lefr_in_lung", "median"),
                                                point_hit=("point_hit", "mean")).round(3).to_string())
rows = []
for f in sorted(glob.glob(os.path.join(OUT, "occlusion", "occ14b__*.npz"))):
    z = np.load(f, allow_pickle=True); _, arch, arm, _ = os.path.basename(f)[:-4].split("__")
    cond = [str(x) for x in z["conditions"]]
    for c in EVAL:
        y, P = z[f"{c}__y"], z[f"{c}__logit"]
        if len(np.unique(y)) < 2: continue
        base = roc_auc_score(y, P[:, 0])
        for j, cc in enumerate(cond):
            a = roc_auc_score(y, P[:, j]); rows.append(dict(arch=arch, arm=arm, cohort=c, condition=cc, auc=a, d_auc=base - a))
O = pd.DataFrame(rows); O.to_csv(os.path.join(OUT, "occlusion_v14b_summary.csv"), index=False)
if len(O):
    print("CHECK arm C, occluding non-lungs (expect dAUC ~0) / lungs (expect large):")
    print(O[(O.arm == "lung") & O.condition.isin(["non_lungs", "lungs"])].pivot_table(index=["arch", "cohort"], columns="condition", values="d_auc").round(3).to_string())
print(f"V14b complete in {(time.time() - T0) / 3600:.2f} h")